# Suite PAR — Parametrics

A parameter is a name a definition declares (`Schemas.OfParameter`), with a description: a symbol standing for what is
bound where the definition is referred to. Parameters are `Parametrics`', normal schemas and one extension, installed
before use ([CAP-112: parameters modeled as schema data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-112.md)): a set of parameters (`Parametrics.OfParameters`), each named by its entry
(`Parametrics.Defines`) and for a value of a schema or an object of an object schema, declared by a definition of any
kind (`parameters`). A schema declares parameters, and a reference object that the schema governs assigns a value to
each parameter for a value and binds each parameter for an object to an object, through the adjacencies
`Parametrics.extend` gives the schema ([CAP-314: parameters bound by the data objects their schema governs](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-314)). Here: installing it, declaring and
validating parameters, snapshots writing them, extending a schema, and assigning and binding parameters.

In [ ]:
import { Builders as B, JSON, Parametrics, Plain, Proxies, Schemas as S, Validators } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, entries_in, equal, given, map, raises, same, spec_of } from "./support.js";

<a id="PAR-01"></a>
## PAR-01 · Parametrics is installed before use: its schemas, a parameter's ends and every kind's set of parameters; installed again, it writes nothing

In [ ]:
{
  const KINDS = ["OfScalar", "OfObject", "OfUnion", "OfIntersection", "OfContainer", "OfAny", "OfRelation", "OfPattern",
    "OfExpression", "OfCollector", "OfModule"];
  const installed: any = new B.Store();
  assert(!installed.names().includes("Parametrics.OfParameters")); // not installed: a store has none
  assert(!installed.Schemas.OfParameter.as_schema().adjacencies.has("for_value"));
  assert(!installed.Schemas.OfObject.as_schema().adjacencies.has("parameters")); // the core declares none
  installed.transaction(Parametrics.install);
  assert(same(installed.names().filter((name: string) => name.startsWith("Parametrics.")), [
    "Parametrics.OfParameters", "Parametrics.Defines", "Parametrics.ForValue", "Parametrics.ForObject",
    "Parametrics.HasParameters", "Parametrics.AssignsValue", "Parametrics.BindsObject"]));
  const [Defines, ForValue, ForObject, HasParameters, AssignsValue, BindsObject] = ["Defines", "ForValue", "ForObject",
    "HasParameters", "AssignsValue", "BindsObject"].map((name) => installed.registered(`Parametrics.${name}`));
  const OfParameters = installed.registered("Parametrics.OfParameters");
  assert(OfParameters.ref && same([...OfParameters.adjacencies.keys()], ["parameters"]));
  assert(same([...Defines.entry.links.keys()], ["parameters", "parameter"]) && same([...Defines.entry.properties.keys()], ["name"]));
  const sets = (relation: any) => relation.uniques.map((unique: Set<string>) => [...unique].sort());
  assert(equal(sets(Defines), [["name", "parameters"], ["parameter"]])); // keyed by name
  assert(equal(sets(ForValue), [["schema"]]) && equal(sets(ForObject), [["schema"]])); // one schema per parameter, shared by many
  assert(equal(sets(HasParameters), [["defn"], ["params"]])); // unique in both directions
  assert(same([...AssignsValue.entry.links.keys()], ["object", "param"])
    && AssignsValue.entry.properties.get("value").schema.schemas.length === 0); // any value
  assert(same([...BindsObject.entry.links.keys()], ["object", "param", "obj_ref"])); // an object binds, not a schema
  const OfParameter = installed.Schemas.OfParameter.as_schema(); // the extension
  assert(equal([...OfParameter.adjacencies].slice(-2).map(([name, a]: any) => [name, a.relation, a.me]),
    [["for_value", ForValue, "param"], ["for_object", ForObject, "param"]]));
  assert(equal(OfParameter.anonymous.slice(-3).map((a: any) => [a.relation, a.me]), [
    [Defines, "parameter"], [AssignsValue, "param"], [BindsObject, "param"]])); // what defines it, and what binds it
  for (const kind of KINDS) {
    const held = installed.Schemas[kind].as_schema().adjacencies.get("parameters");
    assert(held.relation === HasParameters && held.me === "defn");
  }
  const log = JSON.dumps(Plain.ToPlain(installed).Snapshot());
  installed.transaction(Parametrics.install); // installed again: found, nothing written
  assert(JSON.dumps(Plain.ToPlain(installed).Snapshot()) === log);
  assert(!(new B.Store() as any).names().includes("Parametrics.OfParameters")); // each store's own
}

<a id="PAR-02"></a>
## PAR-02 · Every kind of schema declares a set of parameters, each named by its entry, described, and for a value of a schema or an object of an object schema

In [ ]:
/** A schema of every kind declaring a set holding the parameter n, for an int; Matrix, declaring rows and cols; and
 * Tree, whose parameter is for a reference to a tree. */
function declaring(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.declaring/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Object: S.OfObject, Relation: S.OfRelation, Union: S.OfUnion, Intersection: S.OfIntersection,
      Container: S.OfContainer, Pattern: S.OfPattern, Matrix: S.OfObject, Tree: S.OfObject })
    .spec([
      (_: any) => _.Object.name("par.Object").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Relation.name("par.Relation").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Union.name("par.Union").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]).branches([
        (t: any) => t.name("a").schema([S.OfStr]), (t: any) => t.name("b").schema([S.OfStr])]),
      (_: any) => _.Intersection.name("par.Intersection").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]).parts([
        (t: any) => t.name("a").schema([S.OfStr]), (t: any) => t.name("b").schema([S.OfStr])]),
      (_: any) => _.Container.name("par.Container").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]).index([S.OfInt]).element([S.OfStr]),
      (_: any) => _.Pattern.name("par.Pattern").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Matrix.name("Matrix").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        rows: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
        cols: (d: any) => d.parameter((p: any) => p.description("How many columns").for_value([S.OfInt])),
      }))]),
      (_: any) => _.Tree.name("Tree").ref(true).parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        root: (d: any) => d.parameter((p: any) => p.for_object([_.Tree])) }))]),
    ])]))]);
}

{
  const declared: any = new B.Store();
  declared.transaction(Parametrics.install);
  declared.transaction(declaring);
  for (const [name, kind] of [["par.Object", B.OfObject], ["par.Relation", B.OfRelation], ["par.Union", B.OfUnion],
    ["par.Intersection", B.OfIntersection], ["par.Container", B.OfContainer], ["par.Pattern", B.OfPattern]] as const) {
    const defn = declared.registered(name);
    const [held] = [...(new (kind.Builder as any)(defn, false, declared)).parameters]; // one set, its parameters keyed by their names
    assert(same([...held.params.parameters].map((e: any) => e.name.get()), ["n"]) && Parametrics.validate(declared, defn).length === 0);
  }
  const [matrix] = [...new B.OfObject.Builder(declared.registered("Matrix"), false, declared).parameters];
  const [rows, cols] = [...matrix.params.parameters]; // in order
  assert(rows.name.get() === "rows" && cols.name.get() === "cols" && cols.parameter instanceof S.OfParameter.Data);
  assert(rows.parameter.description === null && cols.parameter.description === "How many columns");
  const wide = () => new S.OfParameter.Data({ name: "w", description: "Its width" });
  assert(wide().equals(wide()) && !new S.OfParameter.Data({ name: "w" }).equals(wide())); // by what it holds
  assert(same([...new B.OfParameter.Builder(cols.parameter, false, declared).for_value].map((e: any) => e.schema.name.get()),
    ["Schemas.OfInt"]));
  const [tree] = [...new B.OfObject.Builder(declared.registered("Tree"), false, declared).parameters];
  const [root] = [...tree.params.parameters];
  const ends: any = new B.OfParameter.Builder(root.parameter, false, declared);
  assert(same([...ends.for_object].map((e: any) => e.schema.name.get()), ["Tree"]) && [...ends.for_value].length === 0); // for an object
  const log = JSON.dumps(Plain.ToPlain(declared).Snapshot());
  declared.transaction(declaring); // run again: found, nothing written
  assert(JSON.dumps(Plain.ToPlain(declared).Snapshot()) === log);
  const [Vector] = declared.transaction(() => { // imperatively, the set built where the definition links it
    const Vector = declared.Schemas.OfObject((o: any) => o.name("Vector").parameters([declared.Parametrics.OfParameters(
      (ps: any) => ps.parameters({ n: (d: any) => d.parameter((p: any) => p.for_value([declared.Schemas.OfInt])) }))]));
    return [Vector];
  });
  const [vector] = [...new B.OfObject.Builder(Vector, false, declared).parameters];
  assert(same([...vector.params.parameters].map((e: any) => e.name.get()), ["n"]) && Parametrics.validate(declared, Vector).length === 0);
  assert(Parametrics.validate(declared, declared.Schemas.OfInt.as_schema()).length === 0); // none declared: none to check
}

<a id="PAR-03"></a>
## PAR-03 · Validation checks a set's entries: reserved names, descriptions, a parameter for one of a value and an object, a pattern's names, and one set to one definition

In [ ]:
/** A pattern whose set has a reserved name, a parameter for neither a value nor an object, one for both, and names
 * its symbol and its variable have; and Shared and Twice, which share one set. */
function flawed(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.flawed/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Pattern: S.OfPattern, Node: S.OfObject })
    .spec([
      (_: any) => _.Node.name("par.Node").ref(true),
      (_: any) => _.Pattern.name("par.Flawed").symbols({ x: _.Node }).variables({ w: S.OfInt }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          $n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          x: (d: any) => d.parameter((p: any) => p.for_object([_.Node])),
          w: (d: any) => d.parameter((p: any) => p), // for neither
          b: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt]).for_object([_.Node])), // for both
        }))]),
    ])]))]);
}

{
  const checked: any = new B.Store();
  checked.transaction(Parametrics.install);
  checked.transaction(flawed);
  const Flawed = checked.registered("par.Flawed");
  assert(same(Parametrics.validate(checked, Flawed), [
    "name '$n' is reserved: names starting with '$' belong to the wire format",
    "parameter 'w' is for neither a value nor an object: a parameter is for one",
    "parameter 'b' is for both a value and an object: a parameter is for one",
    "names used as both symbol and parameter: ['x']",
    "names used as both variable and parameter: ['w']"]));
  assert(Flawed.validate().length === 0); // a kind's own validate() does not read the extension's entries
  const [held] = [...new B.OfPattern.Builder(Flawed, false, checked).parameters];
  const dollar = [...held.params.parameters][0].parameter;
  dollar.description = true; // data the builders refuse to build
  assert(Parametrics.validate(checked, Flawed)[1] === "parameter '$n': a description is text, got bool");
  raises(TypeError, () => checked.transaction(() => checked.Schemas.OfParameter((p: any) => p.description(true))),
    "property 'description' holds str, got bool");
  const [One, Two] = checked.transaction(() => { // one set linked by two definitions, and a definition linking two sets
    const shared = checked.Parametrics.OfParameters((ps: any) => ps.parameters({
      n: (d: any) => d.parameter((p: any) => p.for_value([checked.Schemas.OfInt])) }));
    const other = checked.Parametrics.OfParameters((ps: any) => ps);
    const One = checked.Schemas.OfObject((o: any) => o.name("par.One").parameters([shared]));
    const Two = checked.Schemas.OfObject((o: any) => o.name("par.Two").parameters([shared, other]));
    return [One, Two];
  });
  assert(same(Parametrics.validate(checked, One), ["its set of parameters belongs to 2 definitions: a set belongs to one"]));
  assert(same(Parametrics.validate(checked, Two), ["it has 2 sets of parameters: a definition has one",
    "its set of parameters belongs to 2 definitions: a set belongs to one"]));
}

<a id="PAR-04"></a>
## PAR-04 · Snapshots write a definition's set as the extension's entry and read it back; a store without Parametrics ignores what needs it

In [ ]:
/** Grid, declaring rows and cols, each for an int, its cells a container of floats. */
function grid(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.grid/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Grid: S.OfObject })
    .spec([
      (_: any) => _.Grid.name("Grid").parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        rows: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
        cols: (d: any) => d.parameter((p: any) => p.description("Its width").for_value([S.OfInt])),
      }))]).properties({ cells: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfFloat])) }),
    ])]))]);
}

{
  const written: any = new B.Store();
  written.transaction(Parametrics.install);
  written.transaction(grid);
  const log = Plain.ToPlain(written).Snapshot() as any[];
  const [entry] = entries_in(spec_of(log, "Grid", 1), "parameters"); // the extension's entry, with the data: written once
  assert(equal(entry, map({ params: { reference: { symbol: "0" } } })) && entries_in(spec_of(log, "Grid"), "parameters").length === 0);
  const parameters = entries_in(spec_of(log, "0", 1), "parameters"); // the set's, each defining the parameter its entry links
  assert(same(parameters.map((e) => e.get("name")), ["cols", "rows"])); // data's keyed entries, written in the order of their keys
  assert(equal(parameters.map((e) => given(e.get("parameter").get("def").get("mutations"))), [map({ description: "Its width" }), map({})]));
  const back: any = new B.Store();
  back.transaction(Parametrics.install); // a store with Parametrics installed
  Plain.FromPlain(back).Snapshot(log);
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(log)); // read back, written again: the same
  const [held] = [...new B.OfObject.Builder(back.registered("Grid"), false, back).parameters];
  assert(same([...held.params.parameters].map((e: any) => e.name.get()), ["cols", "rows"]));
  assert(Parametrics.validate(back, back.registered("Grid")).length === 0 && back.registered("Grid") !== written.registered("Grid"));
  back.transaction(grid); // its install, run again: found, nothing written
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(log));
  const bare: any = new B.Store(); // a store without it reads it as a snapshot made with extensions it lacks
  const ignored: any[] = [];
  Plain.FromPlain(bare).Snapshot(log, ignored);
  assert(same(ignored.map((i) => i.reason), [...Array(2).fill("Schemas.OfParameter has no adjacency 'for_value'"),
    "Schemas.OfObject has no adjacency 'parameters'"]));
  assert(same([...bare.registered("Grid").properties.keys()], ["cells"]) && !bare.Schemas.OfObject.as_schema().adjacencies.has("parameters"));
}

<a id="PAR-05"></a>
## PAR-05 · `extend` gives a reference object schema the adjacencies through which its objects assign and bind; it refuses a store without Parametrics, a value object schema and a member's name taken; run again, it writes nothing

In [ ]:
/** hw.Bus, a reference object declaring width, for an int, and host, for a hw.Chip; hw.Chip and hw.Spare, reference
 * objects whose schemas declare nothing; hw.Pin, a value object; hw.Taken and hw.Linked, which declare a member of each
 * name extend gives. */
function hardware(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.hardware/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Bus: S.OfObject, Chip: S.OfObject, Spare: S.OfObject, Pin: S.OfObject, Taken: S.OfObject,
      Linked: S.OfObject, Wires: S.OfRelation })
    .spec([
      (_: any) => _.Chip.name("hw.Chip").ref(true), // opts into nothing: any reference object may be bound
      (_: any) => _.Spare.name("hw.Spare").ref(true),
      (_: any) => _.Bus.name("hw.Bus").ref(true).properties({ label: S.OfStr }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          width: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          host: (d: any) => d.parameter((p: any) => p.for_object([_.Chip])),
        }))]),
      (_: any) => _.Pin.name("hw.Pin").properties({ number: S.OfInt }),
      (_: any) => _.Wires.name("hw.Wires").links([(t: any) => t.name("from"), (t: any) => t.name("to")]),
      (_: any) => _.Taken.name("hw.Taken").ref(true).properties({ assigned_values: S.OfStr }),
      (_: any) => _.Linked.name("hw.Linked").ref(true).adjacencies({
        bound_objects: (a: any) => a.relation([_.Wires]).me("from") }),
    ])]))]);
}

const KINDS = ["OfScalar", "OfObject", "OfUnion", "OfIntersection", "OfContainer", "OfAny", "OfRelation", "OfPattern",
  "OfExpression", "OfCollector", "OfModule"];
const bare: any = new B.Store();
raises(LookupError, () => bare.transaction((t: any) => Parametrics.extend(t, [t.Schemas.OfObject])),
  "Parametrics.extend: the store has no Parametrics installed; install it first, "
  + "store.transaction(Parametrics.install)");
const hw: any = new B.Store();
hw.transaction(Parametrics.install);
hw.transaction(hardware);
const [Bus, AssignsValue, BindsObject] = ["hw.Bus", "Parametrics.AssignsValue", "Parametrics.BindsObject"]
  .map((name) => hw.registered(name));
assert(!Bus.adjacencies.has("assigned_values")); // not extended: its objects assign and bind nothing
hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Bus]));
assert(equal([...Bus.adjacencies].map(([name, a]: any) => [name, a.relation, a.me]), [
  ["assigned_values", AssignsValue, "object"], ["bound_objects", BindsObject, "object"]]));
{
  const log = JSON.dumps(Plain.ToPlain(hw).Snapshot());
  hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Bus, hw.registered("hw.Bus")])); // run again: nothing written
  assert(JSON.dumps(Plain.ToPlain(hw).Snapshot()) === log);
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Pin])),
    "Parametrics.extend takes reference object schemas: 'hw.Pin' is a value object schema, "
    + "whose objects are never linked, so that none assigns or binds a parameter");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.Schemas.OfInt])),
    "Parametrics.extend takes reference object schemas: 'Schemas.OfInt' is not one");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Wires])),
    "Parametrics.extend takes reference object schemas: 'hw.Wires' is not one");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, ["nothing"])),
    "Parametrics.extend takes reference object schemas: 'nothing' is not one");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Taken])),
    "Parametrics.extend: 'hw.Taken' already declares a member 'assigned_values'");
  raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Linked])),
    "Parametrics.extend: 'hw.Linked' already declares a member 'bound_objects'");
  assert(JSON.dumps(Plain.ToPlain(hw).Snapshot()) === log); // each refused whole: nothing written
}


<a id="PAR-06"></a>
## PAR-06 · An object assigns a value to a parameter its schema declares, and binds one to an object of any schema, which opts into nothing; a parameter it neither assigns nor binds stays unbound

In [ ]:
hw.transaction(() => {
  const [held] = [...new B.OfObject.Builder(Bus, false, hw).parameters];
  const [width, host] = [...held.params.parameters].map((e: any) => e.parameter);
  const chip = hw.hw.Chip((c: any) => c);
  const bus = hw.hw.Bus((b: any) => b.label("data")
    .assigned_values([(e: any) => e.param(width).value(8n)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(host).obj_ref(chip))));
  const [value] = [...bus.assigned_values];
  const [obj] = [...bus.bound_objects];
  assert(value.param === width && value.value.get() === 8n); // its value an entry property
  assert(obj.param === host && Proxies._durable(obj.obj_ref) === Proxies._durable(chip)); // its object a third link
  const narrow = hw.hw.Bus((b: any) => b.label("narrow").assigned_values([(e: any) => e.param(width).value(1n)]));
  assert([...narrow.bound_objects].length === 0); // host left unbound: it stays unbound
  const loose = hw.hw.Bus((b: any) => b.label("loose"));
  assert([...loose.assigned_values].length === 0 && [...loose.bound_objects].length === 0); // neither is required
  assert([bus, narrow, loose].every((x) => Parametrics.validate(hw, x).length === 0));
  const [chipSchema, spareSchema] = [hw.registered("hw.Chip"), hw.registered("hw.Spare")];
  assert(chipSchema.adjacencies.size === 0 && spareSchema.adjacencies.size === 0 && chipSchema.anonymous.length === 0);
  const spare = hw.hw.Spare((s: any) => s);
  const other = hw.hw.Bus((b: any) => b.label("other").bound_objects((a: any) => a.insert((e: any) => e.param(host).obj_ref(spare))));
  const validate = Validators.Validate(hw);
  assert(validate(Bus, bus).length === 0); // obj_ref links a schema that declares nothing
  assert(same(validate(Bus, other), [ // but a spare is not of host's schema (PAR-13)
    "hw.Bus#0.bound_objects[0].obj_ref: a 'hw.Spare' is bound to a parameter for an object of 'hw.Chip'"]));
  const linked = hw.hw.Linked((x: any) => x.bound_objects((a: any) => a.insert((e: any) => e.to(spare))));
  assert(same(validate(hw.registered("hw.Linked"), linked), [ // another relation's link still asks for an adjacency
    "hw.Linked#0.bound_objects[0].to: a 'hw.Spare' cannot fill link 'to'; its schema declares no adjacency, named or "
    + "anonymous, to this relation via 'to'"]));
  const board = hw.hw.Bus((b: any) => b.label("board"));
  const n = hw.References.OfParameter((p: any) => p.name("n")); // a parameter reference in the place of the parameter
  hw.builder("hw.Bus", board).assigned_values([(e: any) => e.param(n).value(3n)]).update(); // what it assigns is unknown here
  assert(Parametrics.validate(hw, board).length === 0);
});


<a id="PAR-07"></a>
## PAR-07 · An assignment or a binding of a parameter its object's schema does not declare refuses its transaction when it commits; validation reports one already held, and a snapshot reads it back as written

In [ ]:
/** hw.Wire, declaring depth, and hw.Top, a singleton declaring n and m. */
function wiring(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.wiring/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Wire: S.OfObject, Top: S.OfObject })
    .spec([
      (_: any) => _.Wire.name("hw.Wire").ref(true).parameters([txn.Parametrics.OfParameters((ps: any) => ps.parameters({
        depth: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Top.name("hw.Top").ref(true).singleton("hw.Top").parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          m: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
    ])]))]);
}

{
  hw.transaction(wiring);
  hw.transaction((t: any) => Parametrics.extend(t, [t.hw.Wire, t.hw.Top]));
  const log = JSON.dumps(Plain.ToPlain(hw).Snapshot());
  const [depth, free] = hw.transaction(() => {
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Wire"), false, hw).parameters];
    const [depth] = [...held.params.parameters].map((e: any) => e.parameter);
    const free = hw.Schemas.OfParameter((p: any) => p.for_value([hw.Schemas.OfInt])); // in no set
    return [depth, free];
  });
  raises(ValueError, () => hw.transaction(() => {
    hw.hw.Bus((b: any) => b.assigned_values([(e: any) => e.param(depth).value(8n)])); // a hw.Wire's: written, and
  }), "an entry of 'Parametrics.AssignsValue' assigns a value to parameter 'depth', which 'hw.Bus' " // refused at
    + "does not declare: an object assigns and binds only the parameters its schema declares"); // commit, as a unique clause is
  raises(ValueError, () => hw.transaction(() => {
    hw.hw.Bus((b: any) => b.bound_objects((a: any) => a.insert((e: any) => e.param(free).obj_ref(hw.hw.Chip((c: any) => c)))));
  }), "an entry of 'Parametrics.BindsObject' binds a parameter no set of parameters defines to "
    + "an object, which 'hw.Bus' does not declare");
  raises(ValueError, () => hw.transaction(() => { // an assignment names its parameter
    hw.hw.Bus((b: any) => b.assigned_values((a: any) => a.insert((e: any) => e.value(8n))));
  }), "link 'param' is not set");
  assert(JSON.dumps(Plain.ToPlain(hw).Snapshot()) === log); // each refused whole
  hw.transaction(() => {
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Top"), false, hw).parameters];
    const [n] = [...held.params.parameters].map((e: any) => e.parameter);
    hw.builder("hw.Top", hw.singleton("hw.Top")).assigned_values([(e: any) => e.param(n).value(3n)]).update();
  });
  hw.transaction(() => { // an object that assigns one already assigns another: the assignment inserted is checked
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Top"), false, hw).parameters];
    const [, m] = [...held.params.parameters].map((e: any) => e.parameter);
    hw.builder("hw.Top", hw.singleton("hw.Top")).assigned_values((a: any) => a.insert((e: any) => e.param(m).value(4n))).update();
    assert(same([...hw.singleton("hw.Top").assigned_values].map((e: any) => e.value.get()), [3n, 4n]));
  });
  hw.transaction(() => { // the set's declarations removed: the assignments held stay
    const [held] = [...new B.OfObject.Builder(hw.registered("hw.Top"), false, hw).parameters];
    hw.builder("Parametrics.OfParameters", B._linked_data(held.params)).parameters((a: any) => a.clear()).update();
  });
  hw.transaction(() => { // n and m no longer declared: the assignments held are reported, not refused
    assert(same(Parametrics.validate(hw, hw.singleton("hw.Top")), Array(2).fill(
      "an entry of 'Parametrics.AssignsValue' assigns a value to a parameter no set of parameters defines, which "
      + "'hw.Top' does not declare: an object assigns and binds only the parameters its schema declares")));
  });
  const written = Plain.ToPlain(hw).Snapshot(); // the assignments, written as any entries are
  const back: any = new B.Store();
  back.transaction(Parametrics.install);
  Plain.FromPlain(back).Snapshot(written); // read back as written: not refused
  assert(JSON.dumps(Plain.ToPlain(back).Snapshot()) === JSON.dumps(written));
  back.transaction(() => {
    assert(same([...back.singleton("hw.Top").assigned_values].map((e: any) => e.value.get()), [3n, 4n]));
    assert(Parametrics.validate(back, back.singleton("hw.Top")).length === 2);
  });
}


<a id="PAR-08"></a>
## PAR-08 · A schema neither assigns nor binds its parameters: no kind of schema has an adjacency to an assignment or a binding, each links an `object`, and extending a schema of schemas is refused

In [ ]:
for (const kind of KINDS) { // each kind holds its set of parameters, and no assignment or binding
  const schema = hw.Schemas[kind].as_schema();
  assert([...schema.adjacencies.values(), ...schema.anonymous].every((a: any) => a.relation !== AssignsValue
    && a.relation !== BindsObject));
}
assert(same([...AssignsValue.entry.links.keys()], ["object", "param"]) && !BindsObject.entry.links.has("defn"));
raises(TypeError, () => hw.transaction((t: any) => Parametrics.extend(t, [t.Schemas.OfObject])),
  "Parametrics.extend: 'Schemas.OfObject' is a schema of schemas, whose objects are "
  + "schemas: a schema declares its parameters and neither assigns nor binds them");
assert(!hw.Schemas.OfObject.as_schema().adjacencies.has("assigned_values"));


<a id="PAR-09"></a>
## PAR-09 · A parameter context maps a parameter's name to the value assigned to it or the object bound to it; `assign` and `bind` give a sub-context, which maps the name anew and hides the context's own mapping of it, the context unchanged

In [ ]:
/** pcb.Board, a reference object declaring n, for an int, and host, for a pcb.Chip, its size an int, its cells ints,
 * its tag an int as a value of any schema, and the chips it feeds; pcb.Chip, which a board may feed; pcb.Spare, which
 * none may. */
function boards(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.boards/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Board: S.OfObject, Chip: S.OfObject, Spare: S.OfObject, Feeds: S.OfRelation })
    .spec([
      (_: any) => _.Feeds.name("pcb.Feeds").links([(t: any) => t.name("board"), (t: any) => t.name("chip")]),
      (_: any) => _.Chip.name("pcb.Chip").ref(true).adjacencies([(a: any) => a.relation([_.Feeds]).me("chip")]),
      (_: any) => _.Spare.name("pcb.Spare").ref(true),
      (_: any) => _.Board.name("pcb.Board").ref(true).properties({
        size: S.OfInt, cells: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfInt])),
        tag: S.OfAny((a: any) => a.schemas([S.OfInt])) }).adjacencies({
        feeds: (a: any) => a.relation([_.Feeds]).me("board") }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])),
          host: (d: any) => d.parameter((p: any) => p.for_object([_.Chip])),
        }))]),
    ])]))]);
}

const pcb: any = new B.Store();
pcb.transaction(Parametrics.install);
pcb.transaction(boards);
pcb.transaction((t: any) => Parametrics.extend(t, [t.pcb.Board]));
const Board = pcb.registered("pcb.Board");
const [n, host, q] = ["n", "host", "q"].map((name) => pcb.References.OfParameter((p: any) => p.name(name)));
const [validate, check] = [Validators.Validate(pcb), Validators.Check(pcb)];
const empty = new Parametrics.Context();
assert(empty.assigned("n") === null && empty.bound("n") === null); // it maps no name
const target = {}; // an object or a symbol: what is bound is the caller's
const assigned = empty.assign("n", 3n);
const rebound = assigned.bind("n", target);
assert(assigned.assigned("n") === 3n && assigned.bound("n") === null);
assert(rebound.bound("n") === target && rebound.assigned("n") === null); // mapped anew: the outer assignment hidden
assert(empty.assigned("n") === null && assigned.assigned("n") === 3n); // each context unchanged
assert(rebound.assign("m", 4n).bound("n") === target && rebound.assign("m", 4n).assigned("m") === 4n); // other names seen

<a id="PAR-10"></a>
## PAR-10 · Validation checks a parameter reference in the sub-context that the object holding it makes: valid where its parameter is assigned a value, or bound to an object, of the schema its place expects; invalid where it is neither, or is the wrong kind of thing

In [ ]:
const [nParam, hostParam] = pcb.transaction(() => {
  const [held] = [...new B.OfObject.Builder(Board, false, pcb).parameters];
  return [...held.params.parameters].map((e: any) => e.parameter);
});
pcb.transaction(() => {
  const [chip] = [pcb.pcb.Chip((c: any) => c), pcb.pcb.Spare((s: any) => s)];
  const good = pcb.pcb.Board((b: any) => b.size(n).cells(new Map([[1n, n], [2n, 5n]])).tag(n).feeds([(e: any) => e.chip(host)])
    .assigned_values([(e: any) => e.param(nParam).value(3n)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(hostParam).obj_ref(chip))));
  assert(same(validate(Board, good), []) && check(Board, good).holds === true); // each valid: its object assigns or binds it
  const bare = pcb.pcb.Board((b: any) => b.size(n).cells(new Map([[1n, n]])).feeds([(e: any) => e.chip(host)]));
  assert(same(validate(Board, bare), [ // in an empty context: neither assigned nor bound
    "pcb.Board#0.size: parameter 'n' is neither assigned nor bound in its context",
    "pcb.Board#0.cells[1]: parameter 'n' is neither assigned nor bound in its context",
    "pcb.Board#0.feeds[0].chip: parameter 'host' is neither assigned nor bound in its context"]));
  const swapped = pcb.pcb.Board((b: any) => b.size(host).feeds([(e: any) => e.chip(n)])
    .assigned_values([(e: any) => e.param(nParam).value(3n)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(hostParam).obj_ref(chip))));
  assert(same(validate(Board, swapped), [ // an object where a value is expected, and a value where an object is
    "pcb.Board#0.size: parameter 'host' is bound to an object, where a value is expected",
    "pcb.Board#0.feeds[0].chip: parameter 'n' is assigned a value, where an object is expected"]));
  assert(check(Board, swapped).holds === false);
});

<a id="PAR-11"></a>
## PAR-11 · Validation takes a parameter context the caller gives, at every entry point: a reference valid where the context assigns or binds its parameter, of the right schema, and invalid where it maps it to something of another schema; an object's own assignments and bindings hide the caller's

In [ ]:
const [caller, wrong] = pcb.transaction(() => {
  const [chip, spare] = [pcb.pcb.Chip((c: any) => c), pcb.pcb.Spare((s: any) => s)];
  const [good, bare] = ["a", "b"].map(() => pcb.pcb.Board((b: any) => b.size(n).cells(new Map([[1n, n]])).tag(n)
    .feeds([(e: any) => e.chip(host)])));
  pcb.builder("pcb.Board", good).assigned_values([(e: any) => e.param(nParam).value(3n)]).bound_objects(
    (a: any) => a.insert((e: any) => e.param(hostParam).obj_ref(chip))).update();
  const caller = new Parametrics.Context().assign("n", 3n).bind("host", chip);
  assert(same(validate(Board, bare, caller), []) && same(validate.OfObject(Board, bare, caller), [])
    && same(validate.Reachable(Board, bare, caller), []));
  assert(check(Board, bare, caller).holds === true && check.OfObject(Board, bare, caller).holds === true
    && check.Reachable(Board, bare, caller).holds === true);
  const wrong = new Parametrics.Context().assign("n", "three").bind("host", spare);
  assert(same(validate(Board, bare, wrong), [ // the wrong schema: a str for an int, a spare where a chip is fed
    "pcb.Board#0.size: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Board#0.cells[1]: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Board#0.tag: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Board#0.feeds[0].chip: parameter 'host' is bound to an object of another schema: a 'pcb.Spare' cannot fill "
    + "link 'chip'; its schema declares no adjacency, named or anonymous, to this relation via 'chip'"]));
  assert(same(validate(Board, good, wrong), [])); // the object's own assignment and binding hide the caller's
  assert(same(validate(Board, bare, new Parametrics.Context().assign("n", pcb.pcb.Chip((c: any) => c))), [
    "pcb.Board#0.size: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Board#0.cells[1]: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Board#0.tag: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Board#0.feeds[0].chip: parameter 'host' is neither assigned nor bound in its context"]));
  return [caller, wrong];
});
const Int = pcb.Schemas.OfInt;
assert(same(validate(Int, n, caller), []) && same(validate.OfScalar(Int, n, caller), []) && check.OfScalar(Int, n, caller).holds === true);
assert(same(validate.OfScalar(Int, n), ["the value: parameter 'n' is neither assigned nor bound in its context"]));
assert(same(check(Int, n, wrong).problems, ["the value: parameter 'n' is assigned a value of another schema than its place expects"]));
assert(same(validate(Int, 3n, wrong), []) && same(validate(Int, "3"), ["expected int, got str"])); // a scalar: as without a context

<a id="PAR-12"></a>
## PAR-12 · A sub-context follows validation's descent, through each object it checks, and hides an outer name: an object's assignments and bindings map its parameters for its properties and other entries, while its own are checked in the context around it

In [ ]:
/** pcb.Rack, a singleton declaring n, for an int, owning cards by key, each in a slot, an int; pcb.Card, declaring n, for
 * an int, its size an int. */
function chain(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("par.chain/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Rack: S.OfObject, Holds: S.OfRelation, Card: S.OfObject })
    .spec([
      (_: any) => _.Holds.name("pcb.Holds").links([(t: any) => t.name("rack"), (t: any) => t.name("card")]).properties({
        key: S.OfStr, slot: S.OfInt }).uniques([["rack", "key"], ["card"]]),
      (_: any) => _.Card.name("pcb.Card").ref(true).properties({ size: S.OfInt }).adjacencies({
        rack: (a: any) => a.relation([_.Holds]).me("card").owner(true) }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
      (_: any) => _.Rack.name("pcb.Rack").ref(true).singleton("pcb.TheRack").adjacencies({
        cards: (a: any) => a.relation([_.Holds]).me("rack") }).parameters([
        txn.Parametrics.OfParameters((ps: any) => ps.parameters({
          n: (d: any) => d.parameter((p: any) => p.for_value([S.OfInt])) }))]),
    ])]))]);
}

pcb.transaction(chain);
pcb.transaction((t: any) => Parametrics.extend(t, [t.pcb.Rack, t.pcb.Card]));
const [Rack, Card] = [pcb.registered("pcb.Rack"), pcb.registered("pcb.Card")];
const [rackN, cardN] = pcb.transaction(() => [Rack, Card].map((schema) =>
  [...[...new B.OfObject.Builder(schema, false, pcb).parameters][0].params.parameters][0].parameter));
pcb.transaction(() => {
  const rack = pcb.singleton("pcb.TheRack");
  pcb.pcb.Rack(rack, (r: any) => r.assigned_values([(e: any) => e.param(rackN).value("four")]).cards([
    (e: any) => e.key("a").card((c: any) => c.size(n).assigned_values([(e: any) => e.param(cardN).value(4n)])).slot(n),
    (e: any) => e.key("b").card((c: any) => c.size(n)).slot(n)]));
  assert(same(validate.Reachable(Rack, rack, new Parametrics.Context().assign("n", 2n)), [
    "pcb.Rack#0.cards[0].slot: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Rack#0.cards[1].slot: parameter 'n' is assigned a value of another schema than its place expects",
    "pcb.Rack#0.assigned_values[0].value: a value of str is assigned to a parameter for a value of int",
  ])); // the rack's "four" hides the caller's 2 for its entries; card a assigns its own 4; card b sees the caller's 2
  pcb.pcb.Rack(rack, (r: any) => r.assigned_values((a: any) => a.clear().insert((e: any) => e.param(rackN).value(4n))));
  assert(same(validate.Reachable(Rack, rack), [ // what the rack assigns, its entries see; the cards it reaches do not,
    "pcb.Card#2.size: parameter 'n' is neither assigned nor bound in its context", // and an entry seen from card b
    "pcb.Card#2.rack[0].slot: parameter 'n' is neither assigned nor bound in its context"])); // is checked in b's
  const loop = pcb.pcb.Board((b: any) => b.assigned_values([(e: any) => e.param(nParam).value(5n)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(hostParam).obj_ref(host))));
  assert(same(validate(Board, loop), [ // its own binding resolves host in the context around it, not through itself
    "pcb.Board#0.bound_objects[0].obj_ref: parameter 'host' is neither assigned nor bound in its context"]));
});

<a id="PAR-13"></a>
## PAR-13 · Validation checks a parameter assignment's value against its parameter's `ForValue` schema, and a parameter binding's object against its `ForObject` schema: a value of another schema, an object of another schema, a value assigned to a parameter for an object and an object bound to one for a value are each invalid

In [ ]:
pcb.transaction(() => {
  const [chip, spare] = [pcb.pcb.Chip((c: any) => c), pcb.pcb.Spare((s: any) => s)];
  const mixed = pcb.pcb.Board((b: any) => b
    .assigned_values([(e: any) => e.param(nParam).value("three"), (e: any) => e.param(hostParam).value(3n),
      (e: any) => e.param(nParam).value(n)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(hostParam).obj_ref(spare))
      .insert((e: any) => e.param(nParam).obj_ref(chip))));
  assert(same(validate(Board, mixed), [
    "pcb.Board#0.assigned_values[0].value: a value of str is assigned to a parameter for a value of int",
    "pcb.Board#0.assigned_values[1]: a value is assigned to a parameter for an object",
    "pcb.Board#0.assigned_values[2].value: a value of 'References.OfParameter' is assigned to a parameter for a value "
    + "of int", // a parameter reference where a value of any schema is declared: a value of its own schema
    "pcb.Board#0.bound_objects[0].obj_ref: a 'pcb.Spare' is bound to a parameter for an object of 'pcb.Chip'",
    "pcb.Board#0.bound_objects[1]: an object is bound to a parameter for a value"]));
  const unnamed = pcb.pcb.Board((b: any) => b.assigned_values([(e: any) => e.param(q).value("any")]));
  assert(same(validate(Board, unnamed), [ // a parameter reference in the parameter's place: its schema is not known here
    "pcb.Board#0.assigned_values[0].param: parameter 'q' is neither assigned nor bound in its context"]));
  assert(same(validate(Board, unnamed, new Parametrics.Context().bind("q", nParam)), []));
});
hw.transaction(() => { // parameters no set of parameters defines any more (PAR-07): nothing maps them, nor checks them
  assert(same(Validators.Validate(hw)(hw.registered("hw.Top"), hw.singleton("hw.Top")), []));
});

<a id="PAR-14"></a>
## PAR-14 · A parameter binding whose object is a parameter reference resolves it in the context around its object: what that context maps it to maps its parameter, checked against the parameter's `ForObject` schema; where the context maps it to nothing, the reference is invalid, and a reference to the parameter at a link is unknown

In [ ]:
pcb.transaction(() => {
  const [chip, spare] = [pcb.pcb.Chip((c: any) => c), pcb.pcb.Spare((s: any) => s)];
  const relay = pcb.pcb.Board((b: any) => b.feeds([(e: any) => e.chip(host)])
    .bound_objects((a: any) => a.insert((e: any) => e.param(hostParam).obj_ref(q))));
  assert(same(validate(Board, relay, new Parametrics.Context().bind("q", chip)), [])); // host bound to the chip q is bound to
  assert(same(validate(Board, relay, new Parametrics.Context().bind("q", spare)), [
    "pcb.Board#0.feeds[0].chip: parameter 'host' is bound to an object of another schema: a 'pcb.Spare' cannot fill "
    + "link 'chip'; its schema declares no adjacency, named or anonymous, to this relation via 'chip'",
    "pcb.Board#0.bound_objects[0].obj_ref: a 'pcb.Spare' is bound to a parameter for an object of 'pcb.Chip'"]));
  assert(same(validate(Board, relay, new Parametrics.Context().assign("q", 7n)), [
    "pcb.Board#0.feeds[0].chip: parameter 'host' is assigned a value, where an object is expected",
    "pcb.Board#0.bound_objects[0].obj_ref: parameter 'q' is assigned a value, where an object is expected"]));
  const outcome = check(Board, relay); // q mapped to nothing: the binding invalid, and what host stands for unknown
  assert(same(outcome.problems, ["pcb.Board#0.bound_objects[0].obj_ref: parameter 'q' is neither assigned nor bound in "
    + "its context"]));
  assert(same(outcome.unknowns, ["pcb.Board#0.feeds[0].chip: parameter 'host' is bound to parameter 'q', which its context "
    + "neither assigns nor binds"]) && outcome.holds === false);
  pcb.builder("pcb.Board", relay).size(host).update();
  assert(validate(Board, relay)[0] === "pcb.Board#0.size: parameter 'host' is bound to an object, where a value is expected");
});